# Advanced NumPy Algorithms

15 algorithm implementations using NumPy: spiral matrices, sliding windows, convolution, Kadane's 2D, matrix rank, histogram equalization, and more. Each function is standalone and testable.

In [ ]:
import numpy as np

## 1. Spiral Matrix Generator

In [ ]:
def spiral_matrix(n):
    matrix = np.zeros((n, n), dtype=int)
    left, right, top, bottom = 0, n - 1, 0, n - 1
    num = 1
    while left <= right and top <= bottom:
        for i in range(left, right + 1):
            matrix[top][i] = num
            num += 1
        top += 1
        for i in range(top, bottom + 1):
            matrix[i][right] = num
            num += 1
        right -= 1
        for i in range(right, left - 1, -1):
            matrix[bottom][i] = num
            num += 1
        bottom -= 1
        for i in range(bottom, top - 1, -1):
            matrix[i][left] = num
            num += 1
        left += 1
    return matrix

print(spiral_matrix(4))

## 2. Sliding Window Maximum

In [ ]:
def sliding_window_max(arr, k):
    return np.array([np.max(arr[i:i+k]) for i in range(len(arr) - k + 1)])

arr = np.array([1, 3, -1, -3, 5, 3, 6, 7])
print(sliding_window_max(arr, k=3))

## 3. Local Minima and Maxima

In [ ]:
def find_local_minima_maxima(arr):
    minima = (arr[1:-1] < arr[:-2]) & (arr[1:-1] < arr[2:])
    maxima = (arr[1:-1] > arr[:-2]) & (arr[1:-1] > arr[2:])
    return np.where(minima)[0] + 1, np.where(maxima)[0] + 1

arr = np.array([1, 3, 2, 5, 1, 4, 2])
minima, maxima = find_local_minima_maxima(arr)
print("minima indices:", minima)
print("maxima indices:", maxima)

## 4. Custom Padding

In [ ]:
def custom_pad(arr, pad_width, pad_value=0):
    rows, cols = arr.shape
    padded = np.full((rows + 2*pad_width, cols + 2*pad_width), pad_value)
    padded[pad_width:pad_width+rows, pad_width:pad_width+cols] = arr
    return padded

arr = np.array([[1, 2], [3, 4]])
print(custom_pad(arr, pad_width=1, pad_value=0))

## 5. Trimmed Mean (Drop K Lowest Scores per Row)

In [ ]:
def trimmed_row_mean(arr, k):
    sorted_scores = np.sort(arr, axis=1)
    trimmed = sorted_scores[:, k:]
    return np.mean(trimmed, axis=1)

arr = np.array([[5, 1, 8, 3, 7], [2, 9, 4, 6, 5]])
print(trimmed_row_mean(arr, k=2))

## 6. Histogram Equalization

In [ ]:
def hist_equalize(arr):
    hist, bins = np.histogram(arr, bins=256, range=(0, 255), density=True)
    cdf = hist.cumsum()
    cdf_normalized = 255 * cdf / cdf[-1]
    return np.interp(arr, bins[:-1], cdf_normalized).astype(np.uint8)

image = np.random.randint(0, 256, (8, 8), dtype=np.uint8)
equalized = hist_equalize(image.astype(float))
print("original:\n", image)
print("equalized:\n", equalized)

## 7. Diagonal Sorting

In [ ]:
def sort_diagonals(mat):
    mat = mat.copy()
    for offset in range(-mat.shape[0]+1, mat.shape[1]):
        diag = np.diagonal(mat, offset=offset)
        sorted_diag = np.sort(diag)
        if offset >= 0:
            for idx, val in enumerate(sorted_diag):
                mat[idx, idx + offset] = val
        else:
            for idx, val in enumerate(sorted_diag):
                mat[idx - offset, idx] = val
    return mat

mat = np.array([[3, 1, 2], [6, 4, 5], [9, 7, 8]])
print(sort_diagonals(mat))

## 8. 2D Convolution (No scipy/cv2)

In [ ]:
def convolve2d(image, kernel):
    k_h, k_w = kernel.shape
    out_h = image.shape[0] - k_h + 1
    out_w = image.shape[1] - k_w + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            output[i, j] = np.sum(image[i:i+k_h, j:j+k_w] * kernel)
    return output

image = np.array([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]], dtype=float)
kernel = np.array([[1, 0], [0, -1]], dtype=float)
print(convolve2d(image, kernel))

## 9. Frequency Table

In [ ]:
def frequency_table(arr):
    unique, counts = np.unique(arr, return_counts=True)
    return np.vstack((unique, counts)).T

arr = np.array([3, 1, 2, 3, 1, 1, 4, 2, 3])
print(frequency_table(arr))

## 10. Random Block Shuffle (2x2 Blocks)

In [ ]:
def block_shuffle(arr):
    h, w = arr.shape
    blocks = arr.reshape(h//2, 2, w//2, 2).swapaxes(1, 2).reshape(-1, 2, 2)
    np.random.shuffle(blocks)
    return blocks.reshape(h//2, w//2, 2, 2).swapaxes(1, 2).reshape(h, w)

arr = np.arange(1, 17).reshape(4, 4)
print("original:\n", arr)
print("shuffled:\n", block_shuffle(arr))

## 11. Matrix Rank Without np.linalg.matrix_rank

In [ ]:
def manual_rank(matrix):
    mat = matrix.astype(float)
    rows = mat.shape[0]
    rank = 0
    for r in range(rows):
        if not np.any(mat[r]):
            continue
        for i in range(r+1, rows):
            if mat[i, r] != 0:
                mat[[r, i]] = mat[[i, r]]
                break
        if mat[r, r] != 0:
            mat[r] = mat[r] / mat[r, r]
            for i in range(rows):
                if i != r:
                    mat[i] -= mat[i, r] * mat[r]
            rank += 1
    return rank

mat = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("manual rank:", manual_rank(mat))
print("np check:", np.linalg.matrix_rank(mat))

## 12. Maximum 2D Subarray Sum (Kadane's 2D)

In [ ]:
def max_2d_subarray_sum(matrix):
    rows, cols = matrix.shape
    max_sum = float('-inf')
    for top in range(rows):
        temp = np.zeros(cols)
        for bottom in range(top, rows):
            temp += matrix[bottom]
            current = max_current = temp[0]
            for val in temp[1:]:
                current = max(val, current + val)
                max_current = max(max_current, current)
            max_sum = max(max_sum, max_current)
    return max_sum

matrix = np.array([[1, -2, 3], [-4, 5, -6], [7, -8, 9]])
print("max subarray sum:", max_2d_subarray_sum(matrix))

## 13. Array Diff Tracker

In [ ]:
def diff_tracker(arr1, arr2):
    arr1, arr2 = np.array(arr1), np.array(arr2)
    changes = [(i, a, b) for i, (a, b) in enumerate(zip(arr1, arr2)) if a != b]
    added = [(i, b) for i, b in enumerate(arr2) if i >= len(arr1)]
    removed = [(i, a) for i, a in enumerate(arr1) if i >= len(arr2)]
    return {"changes": changes, "added": added, "removed": removed}

a = [1, 2, 3, 4]
b = [1, 9, 3, 4, 5]
print(diff_tracker(a, b))

## 14. Lexicographically Smallest Row

In [ ]:
def smallest_lex_row(arr):
    return arr[np.lexsort(arr.T)[0]]

arr = np.array([[3, 1, 2], [1, 2, 3], [2, 3, 1]])
print("smallest lex row:", smallest_lex_row(arr))

## 15. Mirror Matrix

In [ ]:
def mirror_matrix(arr):
    return np.fliplr(arr), np.flipud(arr)

arr = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
horizontal, vertical = mirror_matrix(arr)
print("horizontal mirror:\n", horizontal)
print("vertical mirror:\n", vertical)